# 05 - Transform: clip, metrics, ensemble stats

Turns the raw subsets into analysis-ready grids:

1. Fetch the TRPA boundary, buffer 10 km (EPSG:26910).
2. Assemble each GCM x scenario LOCA2 series (historical chunks + SSP chunks), convert
   units (`pr` kg m-2 s-1 -> mm/day; `tas*` K -> degC).
3. Per horizon (config) compute: total annual precip, p99 daily precip, max 1-day,
   max 3-day, mean/max tasmax, mean tasmin.
4. Ensemble median / p10 / p90 across GCMs.
5. Clip to the buffered boundary; write NetCDF (lat/lon) + GeoTIFF (EPSG:26910, via
   osgeo.gdal) to `data/processed/`, and a compact CSV of spatial means to `outputs/`.

Runs on whatever subsets exist - missing inputs are logged, not fatal.

In [1]:
import sys
print("Python:", sys.executable)

import warnings
from pathlib import Path
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import xarray as xr

# Pipeline root = climate/ (parent of notebooks/)
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from src.io import load_config, get_logger, append_manifest, sha256_file

cfg = load_config()
log = get_logger("05_transform")

RAW = ROOT / cfg["paths"]["raw"]
PROCESSED = ROOT / cfg["paths"]["processed"]
OUTPUTS = ROOT / cfg["paths"]["outputs"]
for p in (RAW, PROCESSED, OUTPUTS):
    p.mkdir(parents=True, exist_ok=True)

BBOX = cfg["study_area"]["bbox"]
log.info(f"bbox: lon {BBOX['lon_min']}..{BBOX['lon_max']}, lat {BBOX['lat_min']}..{BBOX['lat_max']}")

Python: C:\Program Files\ArcGIS\Pro\bin\Python\envs\arcgispro-py3\python.exe


2026-07-31 22:26:19 | INFO | 05_transform | Log file: C:\Users\mbindl\Documents\GitHub\PROTECT\climate\logs\05_transform_2026-07-31_222619.log


2026-07-31 22:26:19 | INFO | 05_transform | bbox: lon -120.5..-119.5, lat 38.5..39.5


## Boundary + buffer

In [2]:
import geopandas as gpd
import requests as _rq

bnd_json = _rq.get(f"{cfg['study_area']['boundary_url']}/query",
                   params={"where": "1=1", "outFields": "OBJECTID", "f": "geojson"},
                   timeout=120).json()
boundary = gpd.GeoDataFrame.from_features(bnd_json["features"], crs="EPSG:4326")
buffered = boundary.to_crs(epsg=cfg["crs"]["target_epsg"]).buffer(
    cfg["study_area"]["buffer_km"] * 1000).union_all() \
    if hasattr(gpd.GeoSeries, "union_all") else \
    boundary.to_crs(epsg=cfg["crs"]["target_epsg"]).buffer(
    cfg["study_area"]["buffer_km"] * 1000).unary_union
buffered_ll = gpd.GeoSeries([buffered], crs=f"EPSG:{cfg['crs']['target_epsg']}") \
    .to_crs("EPSG:4326").iloc[0]
log.info(f"boundary buffered {cfg['study_area']['buffer_km']} km; "
         f"lat/lon bounds: {[round(b, 3) for b in buffered_ll.bounds]}")

2026-07-31 22:26:33 | INFO | 05_transform | boundary buffered 10 km; lat/lon bounds: [-120.368, 38.614, -119.762, 39.415]


## Assemble LOCA2 series and compute metrics

In [ ]:
from shapely.geometry import Point
from shapely.prepared import prep

L = cfg["sources"]["loca2"]
HORIZONS = cfg["analysis"]["horizons"]
BASE = cfg["analysis"]["baseline"]

# Only the variables the metrics below actually consume. wspeed is extracted for
# later wind metrics but must not gate a model out of the precip/temp ensemble
# (FGOALS-g3 publishes no wind - it still belongs in the 5-model precip/temp set).
METRIC_VARS = ["pr", "tasmax", "tasmin"]

# config.yaml analysis.metrics is AUTHORITATIVE for which metrics ship - this is
# the consultant edit surface. Anything computed below but not configured is
# dropped; a configured metric with no implementation here fails loudly instead
# of silently no-opping (add its formula to precip_metrics()/temp_metrics()).
METRICS_WANTED = list(cfg["analysis"]["metrics"])
IMPLEMENTED_METRICS = {"total_annual_precip", "p99_daily_precip", "max_1day_precip",
                       "max_3day_precip", "mean_tasmax", "max_tasmax", "mean_tasmin"}
_unimplemented = [m for m in METRICS_WANTED if m not in IMPLEMENTED_METRICS]
if _unimplemented:
    raise ValueError(f"config analysis.metrics names metrics with no implementation "
                     f"in 05_transform: {_unimplemented} - add formulas to "
                     f"precip_metrics()/temp_metrics() or remove them from config.yaml")


def keep_configured(mets):
    return {m: g for m, g in mets.items() if m in METRICS_WANTED}

loca2_dir = RAW / "loca2"
files = sorted(loca2_dir.glob("*__tahoe.nc")) if loca2_dir.exists() else []
log.info(f"{len(files)} LOCA2 subset files on disk")


def load_series(gcm, scenario, var):
    """Concat this run's time chunks into one daily series; None if absent.
    Member-agnostic: prefers the configured member, else takes whichever single
    member notebook 01 downloaded (its fallback chain)."""
    preferred = L.get("member_overrides", {}).get(gcm, L["member"])
    pats = sorted(loca2_dir.glob(f"{var}.{gcm}.{scenario}.{preferred}.*__tahoe.nc"))
    if not pats:
        by_member = {}
        for p in sorted(loca2_dir.glob(f"{var}.{gcm}.{scenario}.r*.*__tahoe.nc")):
            by_member.setdefault(p.name.split(".")[3], []).append(p)
        if not by_member:
            return None
        pats = by_member[sorted(by_member)[0]]   # one member only - never mix runs
    ds = xr.concat([xr.open_dataset(p) for p in pats], dim="time").sortby("time")
    da = ds[var] if var in ds else ds[list(ds.data_vars)[0]]
    units = str(da.attrs.get("units", "")).lower()
    if var == "pr" and ("kg" in units or "s-1" in units or "s^-1" in units):
        da = da * 86400.0
        da.attrs["units"] = "mm/day"
    if var.startswith("tas") and ("k" == units or "kelvin" in units or da.max() > 200):
        da = da - 273.15
        da.attrs["units"] = "degC"
    return da


def window(da, y0, y1):
    return da.sel(time=slice(f"{y0}-01-01", f"{y1}-12-31"))


def precip_metrics(da):
    ann = da.resample(time="YS")
    return {
        "total_annual_precip": ann.sum().mean("time"),
        "p99_daily_precip": da.quantile(0.99, dim="time").drop_vars("quantile"),
        "max_1day_precip": ann.max().mean("time"),
        "max_3day_precip": da.rolling(time=3).sum().resample(time="YS").max().mean("time"),
    }


def temp_metrics(da, var):
    ann = da.resample(time="YS")
    out = {f"mean_{var}": da.mean("time")}
    if var == "tasmax":
        out["max_tasmax"] = ann.max().mean("time")
    return out


results = []   # rows: metric, scenario, horizon, gcm -> DataArray
for gcm in L["gcms"]:
    hist = {v: load_series(gcm, "historical", v) for v in METRIC_VARS}
    have = [v for v in METRIC_VARS if hist[v] is not None]
    if not have:
        log.warning(f"[{gcm}] no historical metric variables - skipped")
        continue
    if len(have) < len(METRIC_VARS):
        log.warning(f"[{gcm}] historical missing {sorted(set(METRIC_VARS) - set(have))} "
                    f"- proceeding with {have}")
    # baseline
    for var in have:
        base = window(hist[var], *BASE)
        mets = keep_configured(precip_metrics(base) if var == "pr"
                               else temp_metrics(base, var))
        for m, grid in mets.items():
            results.append({"metric": m, "scenario": "baseline",
                            "horizon": f"{BASE[0]}-{BASE[1]}", "gcm": gcm, "grid": grid})
    # projections: stitch historical tail + ssp so 2020-2049 has its 2015-2020 start
    for scen in L["scenarios"]:
        for var in have:
            proj = load_series(gcm, scen, var)
            if proj is None:
                log.warning(f"[{gcm}] {scen}/{var} missing - skipped")
                continue
            full = xr.concat([hist[var], proj], dim="time").sortby("time")
            for hz in HORIZONS:
                w = window(full, hz["start"], hz["end"])
                mets = keep_configured(precip_metrics(w) if var == "pr"
                                       else temp_metrics(w, var))
                for m, grid in mets.items():
                    results.append({"metric": m, "scenario": scen,
                                    "horizon": hz["name"], "gcm": gcm, "grid": grid})

log.info(f"{len(results)} metric grids computed across "
         f"{len({r['gcm'] for r in results})} GCMs")

## Ensemble stats, clip, write outputs

In [4]:
from osgeo import gdal, osr
gdal.UseExceptions()


def write_geotiff_latlon_then_utm(da, path_utm):
    """da (lat, lon) -> temp lat/lon GeoTIFF -> gdal.Warp to EPSG:26910."""
    lat = da["lat"].values
    lon = da["lon"].values
    lon = np.where(lon > 180, lon - 360, lon)
    arr = da.values
    if lat[0] < lat[-1]:                    # GeoTIFF wants north-up
        arr, lat = arr[::-1, :], lat[::-1]
    dlat = abs(float(lat[0] - lat[1]))
    dlon = abs(float(lon[1] - lon[0]))
    tmp = str(path_utm) + "._ll.tif"
    drv = gdal.GetDriverByName("GTiff")
    dso = drv.Create(tmp, arr.shape[1], arr.shape[0], 1, gdal.GDT_Float32)
    dso.SetGeoTransform([float(lon[0]) - dlon / 2, dlon, 0,
                         float(lat[0]) + dlat / 2, 0, -dlat])
    srs = osr.SpatialReference(); srs.ImportFromEPSG(4326)
    dso.SetProjection(srs.ExportToWkt())
    band = dso.GetRasterBand(1)
    band.WriteArray(np.where(np.isfinite(arr), arr, -9999).astype("float32"))
    band.SetNoDataValue(-9999)
    dso.FlushCache(); dso = None
    gdal.Warp(str(path_utm), tmp, dstSRS=f"EPSG:{cfg['crs']['target_epsg']}",
              creationOptions=["COMPRESS=DEFLATE"])
    Path(tmp).unlink(missing_ok=True)


if results:
    res_df = pd.DataFrame([{k: r[k] for k in ("metric", "scenario", "horizon", "gcm")}
                           for r in results])
    grids = {i: r["grid"] for i, r in enumerate(results)}
    res_df["idx"] = res_df.index

    # clip mask from the first grid's coordinates
    g0 = grids[0]
    lon_vals = np.where(g0["lon"].values > 180, g0["lon"].values - 360, g0["lon"].values)
    prep_b = prep(buffered_ll)
    mask = np.array([[prep_b.contains(Point(x, y)) for x in lon_vals]
                     for y in g0["lat"].values])
    mask_da = xr.DataArray(mask, coords={"lat": g0["lat"], "lon": g0["lon"]},
                           dims=("lat", "lon"))
    log.info(f"clip mask: {int(mask.sum())} of {mask.size} cells inside buffered boundary")

    tif_dir = PROCESSED / "geotiff"
    tif_dir.mkdir(exist_ok=True)
    summary = []
    stats_wanted = cfg["analysis"]["ensemble_stats"]
    for (metric, scen, hz), grp in res_df.groupby(["metric", "scenario", "horizon"]):
        stack = xr.concat([grids[i] for i in grp["idx"]], dim="gcm")
        ens = {"median": stack.median("gcm"),
               "p10": stack.quantile(0.10, "gcm").drop_vars("quantile", errors="ignore"),
               "p90": stack.quantile(0.90, "gcm").drop_vars("quantile", errors="ignore")}
        for stat in stats_wanted:
            da = ens[stat].where(mask_da)
            tag = f"{metric}_{scen}_{hz.replace('-', '_')}_{stat}"
            da.to_netcdf(PROCESSED / f"{tag}.nc")
            if cfg["run"]["write_geotiff"]:
                write_geotiff_latlon_then_utm(da, tif_dir / f"{tag}.tif")
            summary.append({"metric": metric, "scenario": scen, "horizon": hz,
                            "stat": stat, "n_gcms": int(stack.sizes["gcm"]),
                            "spatial_mean": float(da.mean()),
                            "spatial_min": float(da.min()),
                            "spatial_max": float(da.max())})
    sm = pd.DataFrame(summary).sort_values(["metric", "scenario", "horizon", "stat"])
    sm.to_csv(OUTPUTS / "summary_climate_metrics.csv", index=False)
    log.info(f"wrote {len(sm)} summary rows -> outputs/summary_climate_metrics.csv "
             f"+ NetCDF/GeoTIFF per grid in data/processed/")
else:
    log.warning("no LOCA2 subsets available - run 01_extract_loca2 first "
                "(transform skipped gracefully)")

2026-07-31 22:28:30 | INFO | 05_transform | clip mask: 359 of 1024 cells inside buffered boundary


2026-07-31 22:29:33 | INFO | 05_transform | wrote 147 summary rows -> outputs/summary_climate_metrics.csv + NetCDF/GeoTIFF per grid in data/processed/


## Emit web-page payload
Writes `outputs/projections_inline.json` - the compact rows the Projections tab of `html/climate-data.html` embeds. Publication is a deliberate manual step (paste after review), not automatic: the Pages site is public and numbers go up only after a human pass.

In [5]:
import json

sm_path = OUTPUTS / "summary_climate_metrics.csv"
if sm_path.exists():
    sm = pd.read_csv(sm_path)
    rows = sm[["metric", "scenario", "horizon", "stat", "n_gcms", "spatial_mean"]] \
        .round({"spatial_mean": 2}).to_dict("records")
    payload = json.dumps(rows, separators=(",", ":"))
    out = OUTPUTS / "projections_inline.json"
    out.write_text(payload)
    log.info(f"wrote {out.name}: {len(rows)} rows, {len(payload):,} chars")
    log.info("To publish: in html/climate-data.html replace the array after "
             "'const projectionsData = /*__PROJECTIONS__*/' with this file's contents "
             "(review the numbers first - the page is public).")
else:
    log.info("no summary_climate_metrics.csv yet - projections_inline.json not written")

2026-07-31 22:29:33 | INFO | 05_transform | wrote projections_inline.json: 147 rows, 17,086 chars


2026-07-31 22:29:33 | INFO | 05_transform | To publish: in html/climate-data.html replace the array after 'const projectionsData = /*__PROJECTIONS__*/' with this file's contents (review the numbers first - the page is public).
